In [4]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from diffusers import UNet2DModel, DDPMScheduler
from tqdm.auto import tqdm  # Progress bar

# Device configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# DataLoader optimized for Google Colab environment
train_loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,        # Setting to 0 prevents multiprocessing overhead/hangs in Colab
    pin_memory=True
)

# Create U-Net model
model = UNet2DModel(
    sample_size=28,
    in_channels=1,
    out_channels=1,
    layers_per_block=2,
    block_out_channels=(32, 64, 64),
    down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
    up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
).to(device)

# Diffusion scheduler
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

epochs = 5

print("Starting training...")
for epoch in range(epochs):
    model.train()

    # Wrap train_loader in tqdm for a live progress bar
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for images, _ in progress_bar:
        images = images.to(device, non_blocking=True)
        noise = torch.randn_like(images)

        # Sample a random timestep for each image
        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (images.shape[0],),
            device=device
        ).long()

        # Add noise to the clean images
        noisy_images = noise_scheduler.add_noise(images, noise, timesteps)

        # Use Mixed Precision (AMP) to make T4 GPU training much faster
        with torch.amp.autocast(device_type="cuda"):
            noise_pred = model(noisy_images, timesteps).sample
            loss = F.mse_loss(noise_pred, noise)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Update progress bar with current loss metric
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

# Save the trained weights
torch.save(model.state_dict(), "mnist_diffusion_model.pth")
print("Training completed successfully and model saved!")

Using device: cuda
Starting training...


Epoch 1/5:   0%|          | 0/469 [00:00<?, ?it/s]

Epoch 2/5:   0%|          | 0/469 [00:00<?, ?it/s]

Epoch 3/5:   0%|          | 0/469 [00:00<?, ?it/s]

Epoch 4/5:   0%|          | 0/469 [00:00<?, ?it/s]

Epoch 5/5:   0%|          | 0/469 [00:00<?, ?it/s]

Training completed successfully and model saved!
